In [0]:

# 🥉 BRONZE LAYER
# Load all raw files from volume


base_path = "/Volumes/workspace/default/grocery/"

df_orders = spark.read.option("header", "true").option("inferSchema", "true").csv(base_path + "orders.csv")
df_order_products_prior = spark.read.option("header", "true").option("inferSchema", "true").csv(base_path + "order_products__prior.csv")
df_order_products_train = spark.read.option("header", "true").option("inferSchema", "true").csv(base_path + "order_products__train.csv")
df_products = spark.read.option("header", "true").option("inferSchema", "true").csv(base_path + "products.csv")
df_aisles = spark.read.option("header", "true").option("inferSchema", "true").csv(base_path + "aisles.csv")
df_departments = spark.read.option("header", "true").option("inferSchema", "true").csv(base_path + "departments.csv")

print("All 6 files loaded successfully")

All 6 files loaded successfully


In [0]:
#  Inspect schema and count — orders

print(f"orders: {df_orders.count()} rows")
df_orders.printSchema()
display(df_orders.limit(5))

orders: 3421083 rows
root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)



order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
2539329,1,prior,1,2,8,null
2398795,1,prior,2,3,7,15.0
473747,1,prior,3,3,12,21.0
2254736,1,prior,4,4,7,29.0
431534,1,prior,5,4,15,28.0


In [0]:
#  Inspect schema and count — order_products__prior
# ============================================
print(f"order_products__prior: {df_order_products_prior.count()} rows")
df_order_products_prior.printSchema()
display(df_order_products_prior.limit(5))

order_products__prior: 32434489 rows
root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)



order_id,product_id,add_to_cart_order,reordered
2,33120,1,1
2,28985,2,1
2,9327,3,0
2,45918,4,1
2,30035,5,0


In [0]:
# Inspect schema and count — order_products__train

print(f"order_products__train: {df_order_products_train.count()} rows")
df_order_products_train.printSchema()
display(df_order_products_train.limit(5))

order_products__train: 1384617 rows
root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)



order_id,product_id,add_to_cart_order,reordered
1,49302,1,1
1,11109,2,1
1,10246,3,0
1,49683,4,0
1,43633,5,1


In [0]:
# Inspect schema and count — products

print(f"products: {df_products.count()} rows")
df_products.printSchema()
display(df_products.limit(5))

products: 49688 rows
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: string (nullable = true)
 |-- department_id: string (nullable = true)



product_id,product_name,aisle_id,department_id
1,Chocolate Sandwich Cookies,61,19
2,All-Seasons Salt,104,13
3,Robust Golden Unsweetened Oolong Tea,94,7
4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce,38,1
5,Green Chile Anytime Sauce,5,13


In [0]:
#  Inspect schema and count — aisles


print(f"aisles: {df_aisles.count()} rows")
df_aisles.printSchema()
display(df_aisles.limit(5))

aisles: 134 rows
root
 |-- aisle_id: integer (nullable = true)
 |-- aisle: string (nullable = true)



aisle_id,aisle
1,prepared soups salads
2,specialty cheeses
3,energy granola bars
4,instant foods
5,marinades meat preparation


In [0]:
#  Inspect schema and count — departments

print(f"departments: {df_departments.count()} rows")
df_departments.printSchema()
display(df_departments.limit(5))

departments: 21 rows
root
 |-- department_id: integer (nullable = true)
 |-- department: string (nullable = true)



department_id,department
1,frozen
2,other
3,bakery
4,produce
5,alcohol


In [0]:
# Persist all raw tables as Bronze Delta tables

df_orders.write.format("delta").mode("overwrite").saveAsTable("bronze_orders")
df_order_products_prior.write.format("delta").mode("overwrite").saveAsTable("bronze_order_products_prior")
df_order_products_train.write.format("delta").mode("overwrite").saveAsTable("bronze_order_products_train")
df_products.write.format("delta").mode("overwrite").saveAsTable("bronze_products")
df_aisles.write.format("delta").mode("overwrite").saveAsTable("bronze_aisles")
df_departments.write.format("delta").mode("overwrite").saveAsTable("bronze_departments")

print("All 6 Bronze tables created successfully")

All 6 Bronze tables created successfully


In [0]:
# SILVER TABLEE
# Clean and Convert Product Data

from pyspark.sql.functions import expr

df_products = (
    spark.table("bronze_products")
    .withColumn(
        "aisle_id",
        expr("try_cast(aisle_id as INT)")
    )
    .withColumn(
        "department_id",
        expr("try_cast(department_id as INT)")
    )
)

In [0]:
# Join Orders, Products, Aisles, and Departments

df_silver = (
    df_op_prior
    .join(df_orders_prior, "order_id")
    .join(df_products, "product_id", "left")
    .join(df_aisles, "aisle_id", "left")
    .join(df_departments, "department_id", "left")
)

In [0]:
# Data Quality Checks

from pyspark.sql.functions import col, count, when

# Duplicate check
duplicate_count = (
    df_silver
    .groupBy("order_id", "product_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"Duplicate order-product combinations: {duplicate_count}")

# Null check
null_summary = df_silver.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_silver.columns
])

display(null_summary)

Duplicate order-product combinations: 0


department_id,aisle_id,product_id,order_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle,department
3,3,0,0,0,0,0,0,0,0,0,2078068,0,3,3


In [0]:
#  Data Quality Flags

from pyspark.sql.functions import when

df_silver = (
    df_silver
    .withColumn(
        "quality_flag",
        when(col("product_name").isNull(), "Missing Product")
        .when(col("aisle").isNull(), "Missing Aisle")
        .when(col("department").isNull(), "Missing Department")
        .otherwise("Trusted")
    )
)

display(df_silver.groupBy("quality_flag").count())

quality_flag,count
Trusted,32434486
Missing Aisle,3


In [0]:
# Save Silver Table


df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_instacart")

print("Silver table created successfully.")

Silver table created successfully.



Instead of overwriting the Silver table every run, we now MERGE new incoming order data into the existing `silver_instacart` table. This updates matching rows and inserts only new ones, no full rewrite, no duplicates.

In [ ]:
# Incremental Load — Merge new order batch into Silver table
from delta.tables import DeltaTable

# Build the new incoming batch (treating order_products__train as newly arrived orders)
df_orders_new = spark.table("bronze_orders")
df_op_new = spark.table("bronze_order_products_train")

df_silver_new = (
    df_op_new
    .join(df_orders_new, "order_id")
    .join(df_products, "product_id", "left")
    .join(df_aisles, "aisle_id", "left")
    .join(df_departments, "department_id", "left")
    .withColumn(
        "quality_flag",
        when(col("product_name").isNull(), "Missing Product")
        .when(col("aisle").isNull(), "Missing Aisle")
        .when(col("department").isNull(), "Missing Department")
        .otherwise("Trusted")
    )
)

# Point to the existing Silver Delta table
silver_table = DeltaTable.forName(spark, "silver_instacart")

# Merge: update matching order_id + product_id rows, insert new ones, skip duplicates
(
    silver_table.alias("target")
    .merge(
        df_silver_new.alias("source"),
        "target.order_id = source.order_id AND target.product_id = source.product_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)


In [0]:
# GOLD LAYER

# Department-wise Sales
from pyspark.sql.functions import count

df_department_sales = (
    df_silver
    .groupBy("department")
    .agg(count("*").alias("total_products_sold"))
    .orderBy(col("total_products_sold").desc())
)

display(df_department_sales)

department,total_products_sold
produce,9479291
dairy eggs,5414016
snacks,2887550
beverages,2690129
frozen,2236432
pantry,1875577
bakery,1176787
canned goods,1068058
deli,1051249
dry goods pasta,866627


- The produce department is by far the largest driver of volume, with over 9.47 million products sold. 
- dairy eggs holds a strong second place with over 5.41 million products sold, while snacks follows as a popular third category at 2.88 million.

In [0]:
# Top 20 Products
df_top_products = (
    df_silver
    .groupBy("product_name")
    .agg(count("*").alias("times_ordered"))
    .orderBy(col("times_ordered").desc())
)

display(df_top_products.limit(20))

product_name,times_ordered
Banana,472565
Bag of Organic Bananas,379450
Organic Strawberries,264683
Organic Baby Spinach,241921
Organic Hass Avocado,213584
Organic Avocado,176815
Large Lemon,152657
Strawberries,142951
Limes,140627
Organic Whole Milk,137905


- Standard Banana and Bag of Organic Bananas take the top two slots, combining for over 852,000 total orders.
- Out of the top 6 most ordered products, 5 of them are explicitly organic (Organic Bananas, Organic Strawberries, organic Baby Spinach, Organic Hass Avocado, and Organic Avocado). 
- Every single item in the top 9 list belongs to the fresh produce category (bananas, berries, spinach, avocados, and citrus).


In [0]:
# Top Aisles
df_top_aisles = (
    df_silver
    .groupBy("aisle")
    .agg(count("*").alias("orders"))
    .orderBy(col("orders").desc())
)

display(df_top_aisles)

aisle,orders
fresh fruits,3642188
fresh vegetables,3418021
packaged vegetables fruits,1765313
yogurt,1452343
packaged cheese,979763
milk,891015
water seltzer sparkling water,841533
chips pretzels,722470
soy lactosefree,638253
bread,584834


- The fresh fruits and fresh vegetables aisles are neck-and-neck as the most heavily trafficked sections, pulling in over 3.6 million and 3.4 million orders respectively.
- Packaged produce items (packaged vegetables fruits) and daily refrigerator staples (yogurt, packaged cheese, and milk) make up the next massive wave of consumer demand.

In [0]:
# Reorder Analysis
from pyspark.sql.functions import avg

df_reorder = (
    df_silver
    .groupBy("department")
    .agg(
        avg("reordered").alias("reorder_rate")
    )
    .orderBy(col("reorder_rate").desc())
)

display(df_reorder)

department,reorder_rate
dairy eggs,0.6699686517365298
beverages,0.6534601128793452
produce,0.6499125303780631
bakery,0.6281408615152955
deli,0.6077190085317561
pets,0.6012852523433343
babies,0.5789708401564881
bulk,0.577039886616724
snacks,0.5741798410417136
alcohol,0.5699237455756818


- The dairy eggs department leads the store with a remarkable ~67 reorder rate, closely followed by beverages at ~65.3% and produce at ~65%. 
- Items from categories like bakery (62.8%) and deli (60.7%) also show strong repeat purchase patterns, whereas categories like snacks (~57.4%) are slightly lower, indicating customers likely experiment more or buy them less predictably.
